```
# File: windowing.ipynb
# Project: Trabajo de graduación
# Author: María Fernanda Andrade Recinos

The current notebook saves the results to make took identification easier
and to register the information essential for the windowing process.
```

In [22]:
import pandas as pd

# Opciones de visualización de tablas en Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_colwidth', 20)

In [ ]:
"""
# File: windowing.py
# Project: Trabajo de graduación
# Author: María Fernanda Andrade Recinos

Corpus-agnostic windowing for continuous annotations into fixed-size
sections and tags according to a given taxonomy with all the passed in data.
"""


import os
import sys

# 1. Obtenemos el directorio desde donde se está ejecutando el cuaderno
directorio_actual = os.getcwd()

# 2. Verificamos si la carpeta 'implementation' está aquí mismo o un nivel arriba
if os.path.exists(os.path.join(directorio_actual, "implementation")):
    ruta_raiz = directorio_actual
else:
    ruta_raiz = os.path.abspath(os.path.join(directorio_actual, "..", ".."))

# 3. La agregamos al sistema de rutas de Python
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

#print(f"Ruta raíz configurada: {ruta_raiz}")

# Data integration libraries
import pandas as pd
from tabulate import tabulate
from IPython.display import display

from implementation.core.data_loader import build_annotations_index, get_session_data

# Process to analyze data
def split_compound_label(label):
    return set(label.lower().split("_"))

# Merge annotations for a continous range
def merge_annotated_ranges(session_group):
    """
    Merge time intervals to get a continous session according to the annotations
    """
    intervals = sorted(
        zip(session_group["start_time"], session_group["stop_time"])
    )
    merged = []
    for start, end in intervals:
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged

def label_windowing(annotations_df, window_requests, 
                    target_taxonomy, distinguish_taxonomy=None, 
                    exclude_tokens=None, clean_label=None,
                    unreviewd_tokens=False,
                    umbral_artefacto=0.7, umbral_background=0.1):
    rows = []
    group_cols = ["Patient", "Session", "Section", "Montage", "NoChannels", "SamplingFreq", "Duration", "SampledData"]
    window_size_sec = window_requests["window_size_sec"]
    stride_sec = window_requests["stride_sec"]
    win_start_inicial = window_requests.get("win_start", 0)

    for keys, session_group in annotations_df.groupby(group_cols):
        patient, session, section, montage, no_channels, sampling_freq, duration, sampling_data = keys
        win_start = win_start_inicial  # reinicio por grupo

        while (win_start + window_size_sec) <= duration:
            win_end = win_start + window_size_sec

            overlapping = session_group[
                (session_group["start_time"] < win_end) &
                (session_group["stop_time"] > win_start)
            ]

            raw_labels = list(overlapping["label"].unique())
            channels_involved = overlapping["channel"].nunique()

            label_spans = [
                {
                    "label": r.label,
                    "start_in_window": round(max(r.start_time, win_start) - win_start, 3),
                    "end_in_window": round(min(r.stop_time, win_end) - win_start, 3),
                }
                for r in overlapping.itertuples()
            ]

            all_tokens = set()
            for lbl in raw_labels:
                all_tokens |= split_compound_label(lbl)

            row = {
                "Patient": patient, "Session": session,
                "Section": section, "Montage": montage,
                "window_size": window_size_sec, "stride": stride_sec,
                "start": win_start, "end": win_end,
                "raw_labels": raw_labels,
                "label_spans": label_spans,
                "n_channels_annotated": channels_involved,
            }

            # --- Cobertura proporcional por categoría (reemplaza el binario simple) ---
            cobertura = {cat: 0.0 for cat in target_taxonomy}
            for span in label_spans:
                tokens = split_compound_label(span["label"])
                duracion = span["end_in_window"] - span["start_in_window"]
                for cat, keywords in target_taxonomy.items():
                    if tokens & keywords:
                        cobertura[cat] += duracion
            cobertura = {cat: min(v / window_size_sec, 1.0) for cat, v in cobertura.items()}

            for cat in target_taxonomy:
                row[f"coverage_{cat}"] = round(cobertura[cat], 3)

            cobertura_total = sum(cobertura.values())
            if cobertura_total <= umbral_background:
                row["is_clean_window"] = 1
                for cat in target_taxonomy:
                    row[cat] = 0
            elif any(v >= umbral_artefacto for v in cobertura.values()):
                row["is_clean_window"] = 0
                for cat, ratio in cobertura.items():
                    row[cat] = int(ratio >= umbral_artefacto)
            else:
                row["is_clean_window"] = 0
                for cat in target_taxonomy:
                    row[cat] = 0

            row["is_ambiguous"] = int(
                cobertura_total > umbral_background and
                not any(v >= umbral_artefacto for v in cobertura.values())
            )

            row["distinguish"] = int(bool(all_tokens & distinguish_taxonomy)) if distinguish_taxonomy else 0

            genuine_cooccurrence = False
            for lbl in raw_labels:
                tokens = split_compound_label(lbl)
                categories_hit = {g for g, kws in target_taxonomy.items() if tokens & kws}
                if len(categories_hit) > 1:
                    genuine_cooccurrence = True
                    break

            row["genuine_cooccurrence"] = int(genuine_cooccurrence)
            categories_present = sum(row[g] for g in target_taxonomy) + row["distinguish"]
            row["weak_overlap"] = int(categories_present > 1 and not genuine_cooccurrence)

            row["is_unreviewed"] = int(len(all_tokens) == 0)

            if row["is_unreviewed"] and unreviewd_tokens:
                row["is_clean"] = 1
                row["is_unreviewed"] = 0
                row["is_excluded_unreviewed"] = 0
            elif row["is_unreviewed"] and not unreviewd_tokens:
                row["is_clean"] = 0
                row["is_unreviewed"] = 1
                row["is_excluded_unreviewed"] = 1

            row["is_excluded"] = int(bool(all_tokens & exclude_tokens)) if exclude_tokens else 0

            if clean_label is not None:
                has_target = any(row[g] for g in target_taxonomy)
                row["is_clean"] = int(
                    clean_label in all_tokens and not has_target and not row["distinguish"]
                )

            rows.append(row)

            win_start = win_start + stride_sec

    return pd.DataFrame(rows)



if __name__ == "__main__":

    from implementation.core.data_config import ARTIFACT_KEYWORDS, ARTIFACT_ADDITIONAL_TOKENS, BACKGROUND_LABEL, WINDOW_REQUESTS

    database_corpus_pacient = build_annotations_index("artifact", n_patients=1, max_sessions=1)
    display(database_corpus_pacient.head(10))
    test = label_windowing(database_corpus_pacient, WINDOW_REQUESTS["rf_artifact_class"], ARTIFACT_KEYWORDS, unreviewd_tokens=True)
    display(test.head(25))
    #database_windowed = label_windowing(database_corpus_pacient, WINDOW_REQUESTS["rf_artifact_class"], ARTIFACT_KEYWORDS)
    #print(tabulate(database_windowed.head(), headers="keys", tablefmt="psql", showindex=True))



,channel,start_time,stop_time,label,confidence,Patient,Session,Section,Montage,NoChannels,SamplingFreq,Duration,SampledData
0,FP1-F7,22.9737,30.0688,eyem,1.0,aaaaaaju,s005,t000,01_tcp_ar,36,250.0,1441.996,360500
1,FP1-F7,136.7987,140.1117,eyem,1.0,aaaaaaju,s005,t000,01_tcp_ar,36,250.0,1441.996,360500
2,FP1-F7,145.0133,148.0498,eyem,1.0,aaaaaaju,s005,t000,01_tcp_ar,36,250.0,1441.996,360500
3,F7-T3,22.9737,30.0688,eyem,1.0,aaaaaaju,s005,t000,01_tcp_ar,36,250.0,1441.996,360500
4,F7-T3,136.7987,140.1117,eyem,1.0,aaaaaaju,s005,t000,01_tcp_ar,36,250.0,1441.996,360500
5,F7-T3,145.0133,148.0498,eyem,1.0,aaaaaaju,s005,t000,01_tcp_ar,36,250.0,1441.996,360500
6,FP2-F8,22.9737,30.0688,eyem,1.0,aaaaaaju,s005,t000,01_tcp_ar,36,250.0,1441.996,360500
7,FP2-F8,136.7987,140.1117,eyem,1.0,aaaaaaju,s005,t000,01_tcp_ar,36,250.0,1441.996,360500
8,FP2-F8,145.0133,148.0498,eyem,1.0,aaaaaaju,s005,t000,01_tcp_ar,36,250.0,1441.996,360500
9,F8-T4,22.9737,30.0688,eyem,1.0,aaaaaaju,s005,t000,01_tcp_ar,36,250.0,1441.996,360500


,Patient,Session,Section,Montage,window_size,stride,start,end,raw_labels,label_spans,n_channels_annotated,coverage_eye,coverage_muscle,coverage_non_physiological,is_clean_window,eye,muscle,non_physiological,is_ambiguous,distinguish,genuine_cooccurrence,weak_overlap,is_unreviewed,is_clean,is_excluded_unreviewed,is_excluded
0,aaaaaaju,s005,t000,01_tcp_ar,1,1,0,1,[],[],0,0.000,0.0,0.0,1,0,0,0,0,0,0,0,0,1.0,0.0,0
1,aaaaaaju,s005,t000,01_tcp_ar,1,1,1,2,[],[],0,0.000,0.0,0.0,1,0,0,0,0,0,0,0,0,1.0,0.0,0
2,aaaaaaju,s005,t000,01_tcp_ar,1,1,2,3,[],[],0,0.000,0.0,0.0,1,0,0,0,0,0,0,0,0,1.0,0.0,0
3,aaaaaaju,s005,t000,01_tcp_ar,1,1,3,4,[],[],0,0.000,0.0,0.0,1,0,0,0,0,0,0,0,0,1.0,0.0,0
4,aaaaaaju,s005,t000,01_tcp_ar,1,1,4,5,[],[],0,0.000,0.0,0.0,1,0,0,0,0,0,0,0,0,1.0,0.0,0
5,aaaaaaju,s005,t000,01_tcp_ar,1,1,5,6,[],[],0,0.000,0.0,0.0,1,0,0,0,0,0,0,0,0,1.0,0.0,0
6,aaaaaaju,s005,t000,01_tcp_ar,1,1,6,7,[],[],0,0.000,0.0,0.0,1,0,0,0,0,0,0,0,0,1.0,0.0,0
7,aaaaaaju,s005,t000,01_tcp_ar,1,1,7,8,[],[],0,0.000,0.0,0.0,1,0,0,0,0,0,0,0,0,1.0,0.0,0
8,aaaaaaju,s005,t000,01_tcp_ar,1,1,8,9,[],[],0,0.000,0.0,0.0,1,0,0,0,0,0,0,0,0,1.0,0.0,0
9,aaaaaaju,s005,t000,01_tcp_ar,1,1,9,10,[],[],0,0.000,0.0,0.0,1,0,0,0,0,0,0,0,0,1.0,0.0,0


'\nprint("\nTEST label_windowing:")\nresult = label_windowing(\n    database_corpus_pacient,\n    WINDOW_REQUESTS["xgboost_features"],\n    target_taxonomy=ARTIFACT_KEYWORDS,\n    exclude_tokens=ARTIFACT_ADDITIONAL_TOKENS,\n    clean_label=BACKGROUND_LABEL,\n)\ndisplay(result)\nprint(f"\nTotal ventanas: {len(result)}")\nprint(f"is_unreviewed: {result[\'is_unreviewed\'].mean()*100:.2f}%")\nprint(f"is_excluded: {result[\'is_excluded\'].sum()}")\nfor g in ARTIFACT_KEYWORDS:\n    print(f"{g}=1: {result[g].sum()}")\n'